# Steam Game Recommendation: Playtime-Aware BPR

**CSE 258, Assignment 2**

**Authors:** Svante Aretun and Tony Chu

**Research question:** Does playtime-weighted Bayesian Personalized Ranking (BPR) outperform standard BPR on top-K game recommendation?

The notebook follows the five graded sections:

1. Predictive task and evaluation protocol
2. Dataset, preprocessing and exploratory analysis
3. Modeling
4. Evaluation and results
5. Related work and limitations

## 0. Setup

In [ ]:
import ast
import gzip
import json
import math
import os
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
for d in (DATA_DIR, RESULTS_DIR, FIGURES_DIR):
    d.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Predictive task and evaluation protocol

**Task.** Each user owns a set of Steam games, and we know how long they have played each one. We hide some of each user's games and give the model the rest. The model then scores every game the user is not shown as owning, and a good model ranks the hidden games near the top. In other words, the model predicts which games a user owns, not how much they will play them or how they would rate them. This is top-K recommendation from implicit feedback, the setting of Bayesian Personalized Ranking (BPR).

**Research question.** Standard BPR treats every owned game as an equally strong positive. Does weighting each owned game by how long the user played it produce better rankings?

**Evaluation.** For each user, we randomly hide about 10% of their games for validation and another 10% for testing, keeping at least one game in each. The remaining games are the training data. To evaluate, the model ranks every game the user does not already have in training, and we check where the hidden games land. When testing, we also leave out the validation games so they are not counted as candidates. We use four metrics.

- **Recall@10 and Recall@20.** Out of a user's hidden games, how many show up in the model's top 10 or top 20 picks. Users only look at the first few recommendations, so this is what matters most in practice.
- **NDCG@10 (main metric).** Like Recall@10, but a hidden game ranked 1st counts more than one ranked 10th. We use it to pick hyperparameters and to decide when to stop training BPR.
- **AUC.** The chance that a hidden game is ranked above a random game the user does not own. It looks at the whole ranking instead of just the top, and it is what BPR's training objective is built to improve.

Metrics are computed per user and averaged, so heavy users with large libraries do not dominate.

**Baselines.**

1. **Popularity:** recommend the most-owned games to everyone. Any personalized model must beat this.
2. **Item-based Jaccard similarity:** recommend games often owned by the same people as the user's games. A simple, strong collaborative-filtering method with no training.
3. **Standard BPR:** the same model as our proposed method with all weights equal to 1. Comparing against it isolates the effect of playtime weighting.

**Validity.** We pick all hyperparameters using the validation set only. The test set is used only after every modeling choice is fixed. All models use the same split, candidate games and users, with fixed random seeds, so results are fair and reproducible. To make sure the metrics are computed correctly, we run the evaluation code on a tiny example (one user, six games) where the correct answers can be worked out by hand, and confirm the results match (Section 1.1). One limitation applies to every model. A game a user does not own is not necessarily a game they would dislike, so these metrics measure how well a model matches real libraries, not what users truly prefer.

### 1.1 Evaluation functions

These functions are defined early because BPR training in Section 3 uses them for early stopping. Every model has a `score_users(user_ids)` method that returns a score for every game, as an array of shape `(len(user_ids), num_items)`.

`evaluate` takes these scores and computes the metrics from Section 1 for each user. Games the user already has in training are removed from the ranking (validation games are also removed when testing). Ties are broken by lower item ID, so results are the same every run. NDCG@10 gives a hit at rank $r$ a weight of $1/\log_2(r+1)$ and divides by the best possible score, so a perfect ranking scores 1. AUC counts ties as half a win. `sample_eval_users` picks a fixed random subset of users, which makes the early-stopping checks during training faster.

In [ ]:
from scipy.stats import rankdata


def interaction_matrix(dfs, num_users, num_items):
    """Sparse 0/1 user x item matrix from one or more interaction DataFrames."""
    if isinstance(dfs, pd.DataFrame):
        dfs = [dfs]
    users = np.concatenate([df["user_id"].to_numpy() for df in dfs])
    items = np.concatenate([df["item_id"].to_numpy() for df in dfs])
    m = sp.csr_matrix((np.ones(len(users), dtype=np.float32), (users, items)),
                      shape=(num_users, num_items))
    m.data[:] = 1  # duplicates across DataFrames are summed by scipy; reset to 0/1
    return m


def sample_eval_users(eval_df, n_users, seed=SEED):
    """A fixed random subset of users, e.g. for fast per-epoch early stopping."""
    users = np.unique(eval_df["user_id"].to_numpy())
    return np.sort(np.random.default_rng(seed).choice(users, size=min(n_users, len(users)), replace=False))


def evaluate(model, eval_df, mask_dfs, num_users, num_items, ks=(10, 20), ndcg_k=10,
             user_ids=None, batch_size=1024, return_per_user=False):
    """Rank all train-supported items for each eligible user and compute ranking metrics.

    model:     object with score_users(user_ids) -> (len(user_ids), num_items) scores
    eval_df:   held-out interactions (val or test); a user's held-out games are its relevant items
    mask_dfs:  interactions excluded from ranking (train for val; [train, val] for test)
    user_ids:  optional subset of users to evaluate (e.g. from sample_eval_users)

    Eligible users are those with at least one held-out game. For each user:
      Recall@k  = (# held-out games in the top k) / (# held-out games)
      NDCG@10   = DCG@10 / ideal DCG@10, with gain 1 / log2(rank + 1)
      AUC       = probability that a random held-out game is scored above a random
                  candidate the user does not own (ties count 1/2)
    Top-k ties are broken by lower item ID, so results are deterministic.
    Returns macro averages over users, plus per-user metrics if return_per_user=True.
    """
    relevant = interaction_matrix(eval_df, num_users, num_items)
    masked = interaction_matrix(mask_dfs, num_users, num_items)
    assert relevant.multiply(masked).nnz == 0, "held-out items overlap the masked items"

    users = np.flatnonzero(np.diff(relevant.indptr) > 0)  # users with >= 1 held-out game
    if user_ids is not None:
        users = np.intersect1d(users, user_ids)

    max_k = min(max(max(ks), ndcg_k), num_items)
    discounts = 1.0 / np.log2(np.arange(2, max_k + 2))  # gain at rank 1, 2, ...
    ideal_dcg = np.cumsum(discounts[:ndcg_k])

    per_user = []
    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        scores = model.score_users(batch)
        if hasattr(scores, "detach"):  # accept torch tensors
            scores = scores.detach().cpu().numpy()
        scores = np.array(scores, dtype=np.float64)
        assert scores.shape == (len(batch), num_items), scores.shape

        rel = relevant[batch].toarray().astype(bool)
        mask = masked[batch].toarray().astype(bool)
        scores[mask] = -np.inf  # owned (training) games can never be recommended
        n_rel = rel.sum(axis=1)
        n_candidates = num_items - mask.sum(axis=1)

        # Top-k: stable sort on -score, so equal scores keep item-ID order.
        top = np.argsort(-scores, axis=1, kind="stable")[:, :max_k]
        hits = np.take_along_axis(rel, top, axis=1)
        row = {"user_id": batch, "n_relevant": n_rel}
        for k in ks:
            row[f"recall@{k}"] = hits[:, :k].sum(axis=1) / n_rel
        dcg = hits[:, :ndcg_k] @ discounts[:hits[:, :ndcg_k].shape[1]]
        row[f"ndcg@{ndcg_k}"] = dcg / ideal_dcg[np.minimum(n_rel, ndcg_k) - 1]

        # AUC via the Mann-Whitney statistic on average ranks (ties count 1/2).
        # Masked items are -inf, so they take the lowest m ranks; subtracting m gives
        # each item's rank among the candidates only.
        ranks = rankdata(scores, axis=1, method="average") - mask.sum(axis=1, keepdims=True)
        n_neg = n_candidates - n_rel
        rank_sum = np.where(rel, ranks, 0).sum(axis=1)
        row["auc"] = (rank_sum - n_rel * (n_rel + 1) / 2) / (n_rel * n_neg)
        per_user.append(pd.DataFrame(row))

    per_user = pd.concat(per_user, ignore_index=True)
    summary = per_user.drop(columns=["user_id", "n_relevant"]).mean().to_dict()
    summary["n_users"] = len(per_user)
    return (summary, per_user) if return_per_user else summary

In [ ]:
# Sanity check on a toy example small enough to verify by hand.
# One user, 6 games. Game 0 is owned in training (masked); games 1 and 4 are held out.
class FixedScores:
    def __init__(self, scores):
        self.scores = np.asarray(scores, dtype=float)

    def score_users(self, user_ids):
        return self.scores[user_ids]


toy_train = pd.DataFrame({"user_id": [0], "item_id": [0]})
toy_heldout = pd.DataFrame({"user_id": [0, 0], "item_id": [1, 4]})
# Candidate ranking by score: game 1 (rank 1), 2, 3, 4 (rank 4), 5. Game 0 is masked.
toy_model = FixedScores([[9.0, 0.9, 0.8, 0.7, 0.6, 0.5]])
toy = evaluate(toy_model, toy_heldout, [toy_train], num_users=1, num_items=6, ks=(1, 3))

# By hand: NDCG@10 = (1/log2(2) + 1/log2(5)) / (1/log2(2) + 1/log2(3)) = 0.877;
# AUC: game 1 beats all 3 negatives, game 4 beats 1 of 3 -> (3 + 1) / 6 = 0.667;
# Recall@1 = 1/2, Recall@3 = 1/2.
expected = {"recall@1": 0.5, "recall@3": 0.5, "ndcg@10": (1 + 1 / np.log2(5)) / (1 + 1 / np.log2(3)),
            "auc": 4 / 6}
for metric, value in expected.items():
    assert np.isclose(toy[metric], value), (metric, toy[metric], value)
print("Toy example matches hand-computed metrics:", {k: round(float(v), 3) for k, v in expected.items()})

## 2. Dataset, preprocessing and exploratory analysis

**Source.** We use the "User and Item Data" file from Version 1 of the Steam dataset (Pathak, Gupta and McAuley, 2017; Wan and McAuley, 2018; Kang and McAuley, 2018), available from Julian McAuley's dataset page (https://cseweb.ucsd.edu/~jmcauley/datasets.html#steam_data). Steam is Valve's online store and launcher for PC games. Each user has a profile that lists the games they own.

**Contents.** The file has one record per user, with their Steam ID, profile URL and the games they own. For each game it gives the total playtime (`playtime_forever`, in minutes) and the playtime over the last two weeks. We only use the user ID, game ID and total playtime.

**Collection.** Pathak et al. (2017) collected the data by crawling members of "GameAus", an Australian Steam community, and report 88,310 users and 10,978 games. The raw file has the same numbers (88,310 user records and 10,978 games), although some users appear in more than one record (handled in Section 2.2). Only libraries visible to a crawler could be collected, which in practice means public profiles.

**Why this matters for the task.**
- *Owning a game does not mean liking it.* Users often own games they never play, such as games from bundles, gifts, free-to-play titles and sale purchases. A game missing from a library is not a dislike either, since the user may never have seen it. This is why BPR only assumes a user prefers games they own over games they do not, instead of treating unowned games as dislikes.
- *Playtime is a noisy signal of preference.* A few games have huge playtimes, and long, replayable games or games owned for years naturally collect more hours. This is why Section 3.4 tests a log transform and per-user normalization of playtime.
- *The data is a single snapshot.* There are no purchase dates, so we cannot predict future purchases or split the data by time. Instead, we hide a random part of each user's library and ask whether a model can recover it.
- *The users may not be typical.* They all belong to one Australian community group and have public profiles, so they may not represent Steam users in general.

### 2.1 Download and parse

The file has one record per user with a nested list of owned games. We flatten it into one row per (user, game) with the game's lifetime playtime in minutes.

In [ ]:
# The raw file is committed at the repo root; download it only if it is missing.
RAW_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz"
RAW_PATH = Path("australian_users_items.json.gz")

if not RAW_PATH.exists():
    print("Downloading (~74 MB)...")
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)

# Parsing takes ~1.5 minutes, so the flattened tables are cached in data/.
RAW_CACHE = DATA_DIR / "raw_interactions.parquet"
USERS_CACHE = DATA_DIR / "raw_users.parquet"
NAMES_CACHE = DATA_DIR / "item_names.parquet"


def parse_user_items(path):
    """Flatten the nested user -> items records into one row per (user, game).

    Each line of the file is a Python dict literal (single quotes), not strict JSON,
    so it is parsed with ast.literal_eval.
    """
    users, items, playtimes = [], [], []
    user_records = []
    item_names = {}
    with gzip.open(path, "rt") as f:
        for line in f:
            record = ast.literal_eval(line)
            user_records.append((record["user_id"], len(record["items"])))
            for it in record["items"]:
                users.append(record["user_id"])
                items.append(it["item_id"])
                playtimes.append(it["playtime_forever"])
                item_names[it["item_id"]] = it["item_name"]
    raw = pd.DataFrame({"user_id": users, "item_id": items, "playtime_forever": playtimes})
    raw_users = pd.DataFrame(user_records, columns=["user_id", "n_items"])
    names = pd.DataFrame(list(item_names.items()), columns=["item_id", "item_name"])
    return raw, raw_users, names


if RAW_CACHE.exists():
    raw = pd.read_parquet(RAW_CACHE)
    raw_users = pd.read_parquet(USERS_CACHE)
    item_names = pd.read_parquet(NAMES_CACHE)
else:
    raw, raw_users, item_names = parse_user_items(RAW_PATH)
    raw.to_parquet(RAW_CACHE, index=False)
    raw_users.to_parquet(USERS_CACHE, index=False)
    item_names.to_parquet(NAMES_CACHE, index=False)

display(pd.Series({
    "user records in file": len(raw_users),
    "unique users": raw_users["user_id"].nunique(),
    "user records with no games": int((raw_users["n_items"] == 0).sum()),
    "interaction rows": len(raw),
    "unique games": raw["item_id"].nunique(),
}, name="raw data").to_frame())
display(raw.head())

### 2.2 Cleaning and filtering

- **Duplicates.** Some users appear in more than one record, so the same (user, game) pair can show up twice. Total playtime can only go up over time, so we keep the larger value.
- **Zero playtime.** We keep games the user never played. Owning a game still tells us something about the user's interests.
- **Non-game entries.** Some library entries are not real games. These include test and beta builds that Steam adds automatically next to the main game, dedicated servers, SDKs and editors, soundtracks and demos. For example, *Dota 2 Test* is owned by 49,136 users, and 93% of them never played it. Users do not choose these entries, so there is no point modelling or recommending them. They are also almost never played, which would unfairly affect the comparison between standard and playtime-weighted BPR. We remove them based on their names only, never their playtime, and keep a short list of real games whose names happen to match (e.g. *Test Drive Unlimited 2*).
- **Filtering.** We keep users with at least 10 games and games with at least 5 users. Removing a game can drop a user below 10 games, and removing a user can drop a game below 5 users, so we repeat the filter until nothing changes. Users and games with very little data give the models too little to learn from or be evaluated on.

In [ ]:
MIN_GAMES_PER_USER = 10
MIN_USERS_PER_GAME = 5


def summarize(df):
    """Dataset size statistics for one stage of preprocessing."""
    n_users, n_items = df["user_id"].nunique(), df["item_id"].nunique()
    return {
        "users": n_users,
        "games": n_items,
        "interactions": len(df),
        "density": len(df) / (n_users * n_items),
        "zero-playtime fraction": (df["playtime_forever"] == 0).mean(),
    }


# 1. Duplicates: some users appear in more than one record, so the same (user, game)
#    pair can occur twice. playtime_forever is cumulative, so keep the larger value.
pair_playtimes = raw.groupby(["user_id", "item_id"])["playtime_forever"]
n_conflicting = int((pair_playtimes.nunique() > 1).sum())
clean = pair_playtimes.max().reset_index()
print(f"Removed {len(raw) - len(clean):,} duplicate rows "
      f"({n_conflicting} duplicated pairs had different playtimes; kept the max).")

# 2. Validate playtime and convert minutes to hours. Zero-playtime games are kept:
#    owning a game is still an implicit signal even if it was never played.
assert (clean["playtime_forever"] >= 0).all(), "negative playtime found"
clean["playtime_hours"] = clean["playtime_forever"] / 60


# 3. Remove entries that are not standalone games: test/beta builds that Steam grants
#    alongside a game, dedicated servers, SDKs and editors, soundtracks and demos.
#    Owning these is not a choice we want to model or recommend. The filter uses names
#    only (never playtime), with an explicit list of real games whose names match.
NON_GAME_PATTERN = (r"test server|public test|patch testing|system test|\btest\b|\bbeta\b|\bunstable\b"
                    r"|dedicated server|\bsdk\b|\beditor\b|authoring tools|soundtrack|\bdemo\b")
NON_GAME_EXCEPTIONS = {
    "9930",    # Test Drive Unlimited 2
    "211970",  # Test Drive: Ferrari Racing Legends
    "499520",  # The Turing Test
    "430290",  # Shmups Skill Test
    "273550",  # Terrain Test
    "452040",  # GameLoading: Beta The Robot (documentary)
    "490980",  # Daily Chthonicle: Editor's Edition
    "238590",  # Loadout Campaign Beta (standalone free game)
    "270370",  # Lambda Wars Beta (standalone free game)
    "219700",  # Dogs of War Online - Beta (standalone game)
    "313240",  # Wilson Chronicles - Beta (standalone game)
    "457550",  # Bigscreen Beta (standalone app)
}
is_non_game = (item_names["item_name"].str.contains(NON_GAME_PATTERN, case=False, regex=True)
               & ~item_names["item_id"].isin(NON_GAME_EXCEPTIONS))
non_games = item_names.loc[is_non_game, "item_id"]
removed = clean[clean["item_id"].isin(non_games)]
print(f"Removed {non_games.nunique()} non-game entries ({len(removed):,} interactions, "
      f"{len(removed) / len(clean):.1%} of the total). Most-owned removed entries:")
display(removed.merge(item_names, on="item_id").groupby("item_name").size()
        .sort_values(ascending=False).head(10).rename("owners").to_frame())
before_non_game = summarize(clean)
clean = clean[~clean["item_id"].isin(non_games)]


# 4. Iterative k-core filtering: removing games can push users below the threshold
#    and vice versa, so repeat until nothing changes.
def iterative_filter(df, min_games_per_user, min_users_per_game):
    history = [summarize(df)]
    while True:
        n_before = len(df)
        df = df[df.groupby("user_id")["item_id"].transform("size") >= min_games_per_user]
        df = df[df.groupby("item_id")["user_id"].transform("size") >= min_users_per_game]
        history.append(summarize(df))
        if len(df) == n_before:
            break
    return df.reset_index(drop=True), pd.DataFrame(history)


filtered, filter_history = iterative_filter(clean, MIN_GAMES_PER_USER, MIN_USERS_PER_GAME)
filter_history = pd.concat([pd.DataFrame([before_non_game]), filter_history], ignore_index=True)
filter_history.index = (["after dedup", "after removing non-games"]
                        + [f"filter pass {i}" for i in range(1, len(filter_history) - 1)])
display(filter_history)

### 2.3 Train / validation / test split

For each user, we randomly set aside about 10% of their games for validation and another 10% for testing (at least one of each), and use the remaining 80% for training. The random seed is fixed so the split is the same every run. Since the data has no purchase dates, we split at random instead of by time.

Games are numbered using the training split only, because a model cannot score a game it never saw during training. Any validation or test game missing from training would be dropped, but with our filtering this never happens, and every user ends up with at least one validation and one test game.

The split is saved to `data/` so the model sections can load it without rerunning the preprocessing.

In [ ]:
HELDOUT_FRACTION = 0.1  # per user: 10% validation, 10% test, 80% train


def per_user_split(df, seed, heldout_fraction=HELDOUT_FRACTION):
    """Randomly assign each user's games to train/val/test.

    Each user gets n_held = max(1, round(0.1 * n)) validation games and the same number
    of test games; the rest are training games. Users have >= 10 games after filtering,
    so every user has at least 8 training games.
    """
    rng = np.random.default_rng(seed)
    df = df.assign(_r=rng.random(len(df))).sort_values(["user_id", "_r"], kind="stable")
    position = df.groupby("user_id").cumcount().to_numpy()
    n = df.groupby("user_id")["item_id"].transform("size").to_numpy()
    n_held = np.maximum(1, np.floor(heldout_fraction * n + 0.5)).astype(int)
    split = np.where(position < n_held, "test",
                     np.where(position < 2 * n_held, "val", "train"))
    return df.drop(columns="_r").assign(split=split)


split_df = per_user_split(filtered, SEED)

# Contiguous integer IDs. Items are indexed from the training split only: a model cannot
# score a game it never saw in training, so held-out rows for such games are excluded.
user_index = pd.Index(np.sort(split_df["user_id"].unique()))
item_index = pd.Index(np.sort(split_df.loc[split_df["split"] == "train", "item_id"].unique()))

split_df["uid"] = user_index.get_indexer(split_df["user_id"])
split_df["iid"] = item_index.get_indexer(split_df["item_id"])  # -1 = not in training
unsupported = split_df["iid"] < 0
print(f"Excluded {unsupported.sum():,} held-out interactions whose game never appears in training.")
split_df = split_df[~unsupported]


def to_output(part):
    return pd.DataFrame({
        "user_id": part["uid"].astype("int32").to_numpy(),
        "item_id": part["iid"].astype("int32").to_numpy(),
        "playtime_hours": part["playtime_hours"].astype("float32").to_numpy(),
    }).sort_values(["user_id", "item_id"], ignore_index=True)


splits = {name: to_output(split_df[split_df["split"] == name]) for name in ("train", "val", "test")}
num_users, num_items = len(user_index), len(item_index)

# Sanity checks: every user has training data, IDs are contiguous, partitions are disjoint.
assert splits["train"]["user_id"].nunique() == num_users
assert splits["train"]["item_id"].nunique() == num_items
pair_keys = {name: set(zip(s["user_id"], s["item_id"])) for name, s in splits.items()}
assert not (pair_keys["train"] & pair_keys["val"] or pair_keys["train"] & pair_keys["test"]
            or pair_keys["val"] & pair_keys["test"])

total = sum(len(s) for s in splits.values())
display(pd.DataFrame([
    {
        "split": name,
        "interactions": len(s),
        "share": round(len(s) / total, 4),
        "users with >= 1 interaction": s["user_id"].nunique(),
        "users with none (ineligible)": num_users - s["user_id"].nunique(),
    } for name, s in splits.items()
]).set_index("split"))

# Save the shared split (the interface between data/evaluation and model code).
for name, s in splits.items():
    s.to_parquet(DATA_DIR / f"{name}.parquet", index=False)

names_by_id = item_names.set_index("item_id")["item_name"]
meta = {
    "num_users": num_users,
    "num_items": num_items,
    "seed": SEED,
    "heldout_fraction": HELDOUT_FRACTION,
    "min_games_per_user": MIN_GAMES_PER_USER,
    "min_users_per_game": MIN_USERS_PER_GAME,
    "excluded_heldout_interactions": int(unsupported.sum()),
    "user_ids": user_index.tolist(),  # position = integer user_id
    "item_ids": item_index.tolist(),  # position = integer item_id
    "item_names": names_by_id.reindex(item_index).fillna("").tolist(),
}
with open(DATA_DIR / "meta.json", "w") as f:
    json.dump(meta, f)
print(f"Saved train/val/test parquet files and meta.json: {num_users:,} users, {num_items:,} games.")

### 2.4 Exploratory analysis

These statistics use the filtered dataset, with all three splits combined. They are only for understanding the data, and no model uses them. We look at how many games users own and how many users each game has, how playtime is distributed, how the most-owned games compare with the most-played games, and whether users with bigger libraries also play more.

In [ ]:
# Descriptive statistics of the filtered dataset (all splits combined; no model uses them).
eda = filtered.merge(item_names, on="item_id", how="left")
games_per_user = eda.groupby("user_id").size()
users_per_game = eda.groupby("item_id").size()
hours = eda["playtime_hours"]

n_users, n_games = len(games_per_user), len(users_per_game)
display(pd.Series({
    "users": f"{n_users:,}",
    "games": f"{n_games:,}",
    "interactions": f"{len(eda):,}",
    "sparsity": f"{1 - len(eda) / (n_users * n_games):.4%}",
    "games per user (median / mean / max)":
        f"{games_per_user.median():.0f} / {games_per_user.mean():.1f} / {games_per_user.max():,}",
    "users per game (median / mean / max)":
        f"{users_per_game.median():.0f} / {users_per_game.mean():.1f} / {users_per_game.max():,}",
    "zero-playtime fraction": f"{(hours == 0).mean():.1%}",
}, name="filtered dataset").to_frame())

quantiles = [0.25, 0.5, 0.75, 0.9, 0.99, 1.0]
display(pd.DataFrame({
    "all interactions": hours.quantile(quantiles),
    "played (> 0 h)": hours[hours > 0].quantile(quantiles),
}).rename(index=lambda q: "max" if q == 1 else f"{q:.0%}").round(1)
  .rename_axis("playtime quantile (hours)"))

# Most-owned games vs games with the most total playtime: popularity and engagement differ.
by_game = eda.groupby(["item_id", "item_name"]).agg(
    owners=("user_id", "size"),
    total_hours=("playtime_hours", "sum"),
    zero_playtime_share=("playtime_hours", lambda h: (h == 0).mean()),
).reset_index(level=0, drop=True)
top_owned = by_game.sort_values("owners", ascending=False).head(10)
top_played = by_game.sort_values("total_hours", ascending=False).head(10)
display(top_owned.style.format({"total_hours": "{:,.0f}", "zero_playtime_share": "{:.0%}"})
        .set_caption("Top 10 games by number of owners"))
display(top_played.style.format({"total_hours": "{:,.0f}", "zero_playtime_share": "{:.0%}"})
        .set_caption("Top 10 games by total playtime"))

# Does a bigger library mean more total playtime? Relevant to the activity-group
# analysis in Section 4.3, which groups users by total playtime.
user_totals = eda.groupby("user_id")["playtime_hours"].sum()
rho = games_per_user.rank().corr(user_totals.rank())
print(f"Spearman correlation between library size and total playtime: {rho:.2f}")

In [ ]:
import matplotlib.ticker  # noqa: E402  (tick formatting)

# Plot style: one series per panel, recessive axes and grid.
BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
})


def log_hist(ax, values, xlabel, title):
    """Histogram with log-spaced bins on log-log axes, for long-tailed counts."""
    bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 40)
    ax.hist(values, bins=bins, color=BLUE, edgecolor="white", linewidth=0.6)
    ax.set(xscale="log", yscale="log", xlabel=xlabel, ylabel="count", title=title)
    ax.axvline(values.median(), color=INK, linestyle="--", linewidth=1)
    ax.annotate(f"median {values.median():.0f}", xy=(values.median(), 1),
                xycoords=("data", "axes fraction"), xytext=(4, -14),
                textcoords="offset points", color=INK, fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
log_hist(axes[0], games_per_user, "games in library", "Games per user")
log_hist(axes[1], users_per_game, "number of owners", "Users per game")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_degree_distributions.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(hours, bins=60, color=BLUE, edgecolor="white", linewidth=0.6)
axes[0].set(yscale="log", xlabel="playtime (hours)", ylabel="interactions (log scale)",
            title="Raw playtime")
played = hours[hours > 0]
axes[1].hist(np.log1p(played), bins=60, color=BLUE, edgecolor="white", linewidth=0.6)
axes[1].set(xlabel="log(1 + playtime in hours)", ylabel="interactions",
            title="Log-transformed playtime (played games only)")
axes[1].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
axes[1].text(0.98, 0.95, f"not shown: {(hours == 0).sum():,} zero-playtime\ninteractions ({(hours == 0).mean():.0%})",
             transform=axes[1].transAxes, ha="right", va="top", color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_playtime.png", bbox_inches="tight")
plt.show()

**Observations.**

- *A few users and games account for a lot of the data.* The median user owns 52 games, but some own thousands. The median game has 70 owners, while the most popular have over 40,000. This means simply recommending popular games will be a strong baseline, and a good model has to do better than showing everyone the same few games.
- *Playtime is very uneven.* 35% of owned games were never played, the median playtime is under an hour, and the top 1% of entries have over 300 hours. If we weighted the loss by raw hours, a small number of entries would dominate it. Taking `log(1 + t)` spreads the played games out much more evenly, which is why Section 3.4 uses log-scaled weights.
- *Owning a game is not the same as playing it.* The most-owned and most-played lists only partly overlap. *Unturned* and *Portal 2* are among the most owned games but rank much lower by playtime, while *Skyrim* and *Civilization V* rank higher by playtime than by owners. Some free-to-play games are owned by many users who never play them (10% of *Unturned* owners and 18% of *Warframe* owners).
- *Users who own more games also play more.* Library size and total playtime are strongly related (Spearman correlation 0.60), so the activity groups in Section 4.3, which are based on total playtime, partly reflect library size too.

## 3. Modeling

Every model is trained on the training split and gives a score to every game that appears in training. The evaluator then removes the games the user already owns and ranks the rest. Only the weighted BPR models use playtime. The others only use which games each user owns.

| Model | How it scores a game | Cost |
|---|---|---|
| Popularity | The number of training users who own it. | One pass over the data. Every user gets the same ranking (before their own games are removed). |
| Item-based Jaccard | How similar the game is to each game the user owns, based on how many users own both, added up. | Similarities are computed in blocks of games to limit memory. Keeping every neighbor means storing a full game-by-game matrix. |
| BPR | The dot product of a user embedding and a game embedding. Training pushes each owned game to score higher than a randomly sampled game the user does not own. | Needs many training passes, with a validation check after each one. Stores one vector per user and per game. |

We compare three versions of BPR: standard BPR, log-weighted BPR and user-normalized BPR. All three use the same model and training settings. The only difference is how much each owned game counts in the loss, which is equal for standard BPR and depends on playtime for the other two (Section 3.4).

### 3.0 Load the shared split

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
with open(DATA_DIR / "meta.json") as f:
    meta = json.load(f)
num_users, num_items = meta["num_users"], meta["num_items"]

### 3.1 Popularity baseline

Each game is scored by how many training users own it, so every user gets the same ranking, apart from the games they already own being removed. It ignores personal taste completely, which makes it the simplest baseline that any personalized model should beat.

This cell also picks a fixed random set of 5,000 validation users, which Section 3.2 uses to choose the number of Jaccard neighbors quickly.

In [ ]:
class Popularity:
    """Score every game by its number of distinct training users (the same list for everyone)."""

    def __init__(self, train_df, num_items):
        self.counts = np.bincount(train_df["item_id"], minlength=num_items).astype(np.float32)

    def score_users(self, user_ids):
        return np.broadcast_to(self.counts, (len(user_ids), len(self.counts)))


popularity = Popularity(train_df, num_items)
EARLY_STOP_USERS = sample_eval_users(val_df, 5000)  # fixed subset for quick validation checks
print("Popularity (validation, 5,000 users):",
      {k: round(v, 4) for k, v in evaluate(popularity, val_df, [train_df], num_users, num_items,
                                           user_ids=EARLY_STOP_USERS).items()})

### 3.2 Item-based Jaccard similarity

The similarity between two games is $J(i,j)=\frac{|U_i\cap U_j|}{|U_i\cup U_j|}$, where $U_i$ is the set of training users who own game $i$. In words, it is the number of users who own both games divided by the number of users who own either. A candidate game's score is the sum of its similarities to the games the user owns in training, so games that are often owned by the same people as the user's games rank highly.

The game-by-game similarities are computed one block of games at a time using sparse matrix products, which keeps memory use manageable. Each game can either keep only its $k$ most similar games (ties broken by lower game ID) or keep all of them. We choose $k$ on the validation users below.

In [ ]:
class ItemJaccard:
    """Item-based collaborative filtering with Jaccard similarity.

    J(i, j) = |U_i & U_j| / |U_i | U_j|, where U_i is the set of training users who own game i.
    Each game keeps only its `top_k_neighbors` most similar games (None keeps all; ties broken
    by lower game ID). A user's score for game j is the sum of J(i, j) over the games i in
    their training library that kept j as a neighbour.
    """

    def __init__(self, train_df, num_users, num_items, top_k_neighbors=100, block_size=1024):
        # items x users ownership matrix
        X = sp.csr_matrix((np.ones(len(train_df), dtype=np.float32),
                           (train_df["item_id"], train_df["user_id"])), shape=(num_items, num_users))
        n_owners = np.asarray(X.sum(axis=1)).ravel()
        XT = X.T.tocsc()
        prune = top_k_neighbors is not None and top_k_neighbors < num_items
        # Pruned: sparse (top-k per row). All neighbours: ~80% of pairs are nonzero, so dense
        # float32 (num_items^2 * 4 bytes, ~330 MB) is smaller and faster than sparse.
        S = None if prune else np.zeros((num_items, num_items), dtype=np.float32)
        rows, cols, vals = [], [], []
        # Compute similarities one block of games at a time to bound memory.
        for start in range(0, num_items, block_size):
            block = np.arange(start, min(start + block_size, num_items))
            co = (X[block] @ XT).toarray()  # |U_i & U_j|
            sim = co / (n_owners[block, None] + n_owners[None, :] - co)  # Jaccard
            sim[np.arange(len(block)), block] = 0  # a game is not its own neighbour
            if prune:
                keep = np.argsort(-sim, axis=1, kind="stable")[:, :top_k_neighbors]
                kept = np.take_along_axis(sim, keep, axis=1)
                nonzero = kept > 0
                rows.append(np.repeat(block, nonzero.sum(axis=1)))
                cols.append(keep[nonzero])
                vals.append(kept[nonzero].astype(np.float32))
            else:
                S[block] = sim
        if prune:  # S[i, j] = J(i, j) if j is among game i's kept neighbours, else 0
            S = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                              shape=(num_items, num_items))
        self.S = S
        self.R = sp.csr_matrix((np.ones(len(train_df), dtype=np.float32),
                                (train_df["user_id"], train_df["item_id"])), shape=(num_users, num_items))

    def score_users(self, user_ids):
        scores = self.R[user_ids] @ self.S  # sum of similarities to the user's games
        return scores.toarray() if sp.issparse(scores) else np.asarray(scores)

In [ ]:
# Choose the neighbourhood size on validation data (fixed user subset).
jaccard_sweep = []
for k in (25, 100, 400, None):
    t0 = time.time()
    model = ItemJaccard(train_df, num_users, num_items, top_k_neighbors=k)
    fit_seconds = time.time() - t0
    metrics = evaluate(model, val_df, [train_df], num_users, num_items, user_ids=EARLY_STOP_USERS)
    jaccard_sweep.append({"top_k_neighbors": "all" if k is None else k, **metrics,
                          "fit seconds": round(fit_seconds, 1)})
jaccard_sweep = pd.DataFrame(jaccard_sweep).set_index("top_k_neighbors").drop(columns="n_users")
display(jaccard_sweep.round(4))

best_k = jaccard_sweep["ndcg@10"].idxmax()
jaccard = ItemJaccard(train_df, num_users, num_items,
                      top_k_neighbors=None if best_k == "all" else int(best_k))
print(f"Selected top_k_neighbors = {best_k} (highest validation NDCG@10).")

### 3.3 Standard BPR

For user $u$ and game $i$, the score is the dot product $s(u,i)=p_u^\top q_i$. Each epoch we shuffle the training interactions. For each owned game, we draw one negative uniformly from the games outside that user's training library. If a user owns the whole training catalog, there is nothing to sample for that user.

We use Adam to minimize the average loss over these triplets. The softplus term gets smaller as the owned game scores higher than the sampled game. The second term penalizes large values in the three embeddings.

$$
\mathcal L = \mathbb E_{(u,i,j)}\left[
\operatorname{softplus}\big(s(u,j)-s(u,i)\big)
+ \lambda\big(\|p_u\|_2^2 + \|q_i\|_2^2 + \|q_j\|_2^2\big)
\right].
$$

`train_bpr` returns the fitted model and its training history. With a validation split, we keep the checkpoint with the highest NDCG@10. The default patience is five epochs. `score_users` returns the scores in item-ID order so the evaluator can mask the user's known games.

In [ ]:
class BPR(torch.nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        self.user_embeddings = torch.nn.Embedding(num_users, dim)
        self.item_embeddings = torch.nn.Embedding(num_items, dim)
        torch.nn.init.normal_(self.user_embeddings.weight, std=0.01)
        torch.nn.init.normal_(self.item_embeddings.weight, std=0.01)

    @torch.no_grad()
    def score_users(self, user_ids):
        """Score every item in the shared item-ID order."""
        ids = torch.as_tensor(user_ids, dtype=torch.long,
                              device=self.user_embeddings.weight.device)
        scores = self.user_embeddings(ids) @ self.item_embeddings.weight.T
        return scores.cpu().numpy()


def _bpr_validation_scorer(train_df, val_df, num_users, num_items, batch_size=512):
    """Compute the shared evaluator's NDCG@10 on every eligible user."""
    relevant = interaction_matrix(val_df, num_users, num_items).astype(bool)
    masked = interaction_matrix(train_df, num_users, num_items)
    if relevant.multiply(masked).nnz:
        raise ValueError("Validation interactions overlap the training mask.")

    counts = np.diff(relevant.indptr)
    users = np.flatnonzero(counts)
    if not len(users):
        raise ValueError("BPR checkpoint selection requires validation interactions.")
    k = min(10, num_items)
    discounts = 1 / np.log2(np.arange(2, k + 2))
    ideal = np.cumsum(discounts)[np.minimum(counts[users], k) - 1]
    batches = []
    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        batches.append((batch, masked[batch].nonzero(), relevant[batch]))

    def ndcg_at_10(model):
        dcg = np.empty(len(users), dtype=np.float64)
        for start, (batch, mask, truth) in enumerate(batches):
            scores = model.score_users(batch)
            scores[mask] = -np.inf
            values, indices = torch.topk(torch.from_numpy(scores), min(k + 1, num_items))
            values, top = values.numpy(), indices.numpy()[:, :k]
            # Look one spot past the cutoff to catch a tie for the last spot.
            tied_rows = np.flatnonzero((values[:, 1:] == values[:, :-1]).any(axis=1))
            for row in tied_rows:
                top[row] = np.argsort(-scores[row], kind="stable")[:k]
            hits = np.take_along_axis(truth.toarray(), top, axis=1)
            offset = start * batch_size
            dcg[offset:offset + len(batch)] = hits @ discounts
        return float(np.mean(dcg / ideal))

    return ndcg_at_10


def train_bpr(train_df, val_df, num_users, num_items, weights=None, dim=64, lr=1e-3,
              reg=1e-4, batch_size=4096, epochs=30, seed=SEED, patience=5, verbose=False, fused_adam=False):
    """Return the model and per-epoch training history.

    Select checkpoints by full-catalog validation NDCG@10. Pass val_df=None
    to train for a fixed number of epochs.
    """
    users = train_df["user_id"].to_numpy(dtype=np.int64)
    items = train_df["item_id"].to_numpy(dtype=np.int64)
    if not len(users):
        raise ValueError("BPR requires training interactions.")
    if (users.min() < 0 or users.max() >= num_users
            or items.min() < 0 or items.max() >= num_items):
        raise ValueError("Training IDs must be within the shared ID ranges.")
    if min(dim, batch_size, epochs, patience) < 1:
        raise ValueError("dim, batch_size, epochs and patience must be positive.")
    if weights is None:
        weights = np.ones(len(users), dtype=np.float32)
    else:
        weights = np.asarray(weights, dtype=np.float32)
        if (weights.shape != users.shape or not np.isfinite(weights).all()
                or np.any(weights < 0)):
            raise ValueError("Provide one finite, nonnegative weight per training row.")

    # Draw negatives from the games seen in training. Scores still use
    # the original item IDs.
    catalog, item_ranks = np.unique(items, return_inverse=True)
    catalog_size = len(catalog)
    seen = np.unique(users * catalog_size + item_ranks)
    seen_users = seen // catalog_size
    seen_counts = np.bincount(seen_users, minlength=num_users)
    row_starts = np.cumsum(seen_counts) - seen_counts
    available = catalog_size - seen_counts
    eligible_rows = np.flatnonzero(available[users])
    if not len(eligible_rows):
        raise ValueError("No training user has an unowned game to sample.")

    # Within a user's row, each gap key counts the unowned games before
    # an owned game. The search tells us how many owned games to skip.
    seen_offsets = np.arange(len(seen)) - row_starts[seen_users]
    gap_keys = seen - seen_offsets

    sampling_rng = np.random.default_rng(seed)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(seed)
        model = BPR(num_users, num_items, dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, fused=fused_adam)

    score_validation = (
        _bpr_validation_scorer(train_df, val_df, num_users, num_items)
        if val_df is not None else None
    )
    history = []
    best_score = -np.inf
    best_state = None
    stale_epochs = 0
    for epoch in range(1, epochs + 1):
        model.train()
        shuffled_rows = sampling_rng.permutation(eligible_rows)
        total_loss = 0.0
        for start in range(0, len(shuffled_rows), batch_size):
            rows = shuffled_rows[start:start + batch_size]
            batch_users = users[rows]
            draws = sampling_rng.integers(available[batch_users])
            queries = batch_users * catalog_size + draws
            skipped = np.searchsorted(gap_keys, queries, side="right")
            negative_ranks = draws + skipped - row_starts[batch_users]

            user_vectors = model.user_embeddings(torch.from_numpy(batch_users))
            positive_vectors = model.item_embeddings(torch.from_numpy(items[rows]))
            negative_vectors = model.item_embeddings(torch.from_numpy(catalog[negative_ranks]))
            margins = (user_vectors * (positive_vectors - negative_vectors)).sum(dim=1)
            ranking_loss = (torch.from_numpy(weights[rows])
                            * torch.nn.functional.softplus(-margins)).mean()
            penalty = (user_vectors.square().sum(dim=1)
                       + positive_vectors.square().sum(dim=1)
                       + negative_vectors.square().sum(dim=1)).mean()
            loss = ranking_loss + reg * penalty

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(rows)

        record = {"epoch": epoch, "train_loss": total_loss / len(shuffled_rows)}
        history.append(record)
        if score_validation is not None:
            model.eval()
            score = score_validation(model)
            if not np.isfinite(score):
                raise ValueError("Validation NDCG@10 must be finite.")
            record["val_ndcg@10"] = score
            if verbose:
                print(f"Epoch {epoch:2d} | loss {record['train_loss']:.6f} | "
                      f"validation NDCG@10 {score:.6f}", flush=True)
            if score > best_score:
                best_score = score
                best_state = {name: value.clone() for name, value in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
                if stale_epochs >= patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.eval(), history

### 3.4 Playtime-weighted BPR

Let $t_{ui}$ be the number of hours user $u$ played game $i$ in the training data. We use $a_{ui}=(1+\log(1+t_{ui}))^\alpha$ to give more weight to games played for longer. The logarithm compresses very large playtimes. We multiply the softplus term in Section 3.3 by the resulting weight $w_{ui}$ and keep the same embeddings, sampler, and L2 penalty.

| Variant | Weight |
|---|---|
| Standard BPR | 1 |
| Log-weighted BPR | $a_{ui}$ |
| User-normalized BPR | $a_{ui}/\operatorname{mean}_{k\in I_u^{train}}a_{uk}$ |

We rescale the weights so the average over eligible training interactions is one. User normalization also keeps the average at one within each person's library, so each user's total weight matches standard BPR. A game with zero recorded hours starts with a raw weight of one. At $\alpha=0$, every weight is one and we recover standard BPR.

In [ ]:
def playtime_weights(train_df, alpha=1.0, user_normalize=False):
    """Return row-aligned weights with mean one over trainable triplets."""
    hours = train_df["playtime_hours"].to_numpy(dtype=np.float64)
    if not np.isfinite(alpha) or alpha < 0:
        raise ValueError("alpha must be finite and nonnegative.")
    if not np.isfinite(hours).all() or np.any(hours < 0):
        raise ValueError("Training playtime must be finite and nonnegative.")

    weights = np.ones(len(hours), dtype=np.float32)
    if alpha == 0 or not len(hours):
        return weights

    # Shift the log weights before taking exp to avoid overflow.
    # Dividing by the mean cancels out that shift.
    log_base = np.log1p(np.log1p(hours))
    if user_normalize:
        users = train_df["user_id"].to_numpy()
        maxima = pd.Series(log_base).groupby(users, sort=False).transform("max").to_numpy()
        scaled = np.exp(alpha * (log_base - maxima))
        means = pd.Series(scaled).groupby(users, sort=False).transform("mean").to_numpy()
        return (scaled / means).astype(np.float32)

    owned = train_df.groupby("user_id", sort=False)["item_id"].transform("nunique")
    eligible = owned.to_numpy() < train_df["item_id"].nunique()
    if eligible.any():
        eligible_log_base = log_base[eligible]
        scaled = np.exp(alpha * (eligible_log_base - eligible_log_base.max()))
        weights[eligible] = scaled / scaled.mean()
    return weights


def run_alpha_sweep(train_df, val_df, num_users, num_items,
                    alphas=(0, 0.25, 0.5, 1, 2), previous_runs=None, **bpr_config):
    """Return epoch records, the selected alpha, and its fitted model."""
    records = []
    best_alpha, best_model, best_score = None, None, -np.inf
    previous_runs = {} if previous_runs is None else previous_runs
    for alpha in sorted(alphas):
        if alpha in previous_runs:
            model, history = previous_runs[alpha]
        else:
            weights = playtime_weights(train_df, alpha=alpha)
            model, history = train_bpr(
                train_df, val_df, num_users, num_items, weights=weights, **bpr_config
            )
        run_best = max(history, key=lambda row: row["val_ndcg@10"])
        if run_best["val_ndcg@10"] > best_score:
            best_alpha, best_model, best_score = alpha, model, run_best["val_ndcg@10"]
        records.extend({
            "model": "standard_bpr" if alpha == 0 else "log_weighted_bpr",
            "split": "validation",
            "alpha": alpha,
            **bpr_config,
            **row,
        } for row in history)
    return pd.DataFrame(records), best_alpha, best_model

## 4. Evaluation and results

We use the metrics defined in Section 1: Recall@10, Recall@20, NDCG@10 and AUC. For the test results, each user's training and validation games are removed from the ranking, and the metrics are averaged over all users (every user has at least one test game). All checkpoints and hyperparameters were chosen on validation data before the test set was used.

### 4.1 Experiment A: main comparison (test set)

Each BPR variant was trained for up to 100 epochs, stopping once validation NDCG@10 had not improved for five epochs, and we kept the best checkpoint. For Jaccard, keeping all neighbors gave the highest NDCG@10 on the 5,000 validation users. All of these choices were made on validation data before the test set was scored.

Standard BPR had the best test score on all four metrics, with NDCG@10 of 0.2790, Recall@10 of 0.2603, Recall@20 of 0.3567 and AUC of 0.9772. Jaccard came second on NDCG@10 at 0.2702. Both weighted variants scored lower than standard BPR, at 0.2586 for log weighting and 0.2524 for user normalization. These results come from a single random seed.

In [ ]:
def evaluate_test_models(models, train_df, val_df, test_df, num_users, num_items):
    """Evaluate fixed models on the same test users and candidate masks."""
    rows, per_user = [], {}
    for name, (model, configuration) in models.items():
        metrics, per_user[name] = evaluate(
            model, test_df, [train_df, val_df], num_users, num_items,
            batch_size=512, return_per_user=True,
        )
        rows.append({
            "model": name,
            "seed": SEED,
            "split": "test",
            "config": json.dumps(configuration, sort_keys=True),
            **metrics,
        })
    return pd.DataFrame(rows), per_user


torch.set_num_threads(2)
comparison_bpr_config = dict(dim=64, lr=1e-3, reg=1e-4, batch_size=4096,
                             epochs=100, seed=SEED, patience=5, fused_adam=True)
bpr_variants = {
    "standard_bpr": (0, False),
    "log_weighted_bpr": (1, False),
    "user_normalized_bpr": (1, True),
}
bpr_runs = {}
test_models = {
    "popularity": (popularity, {}),
    "item_jaccard": (jaccard, {"top_k_neighbors": best_k}),
}
for name, (alpha, normalize) in bpr_variants.items():
    weights = playtime_weights(train_df, alpha=alpha, user_normalize=normalize)
    bpr_runs[name] = train_bpr(
        train_df, val_df, num_users, num_items, weights=weights, **comparison_bpr_config
    )
    model, history = bpr_runs[name]
    best_epoch = max(history, key=lambda row: row["val_ndcg@10"])["epoch"]
    configuration = dict(alpha=alpha, user_normalize=normalize, checkpoint_epoch=best_epoch,
                         epochs_run=len(history),
                         **comparison_bpr_config)
    test_models[name] = model, configuration

alpha_history, best_alpha, best_alpha_model = run_alpha_sweep(
    train_df, val_df, num_users, num_items,
    previous_runs={0: bpr_runs["standard_bpr"], 1: bpr_runs["log_weighted_bpr"]},
    **comparison_bpr_config,
)
if best_alpha not in (0, 1):
    selected_rows = alpha_history.loc[alpha_history["alpha"] == best_alpha]
    best_epoch = int(selected_rows.loc[selected_rows["val_ndcg@10"].idxmax(), "epoch"])
    configuration = dict(alpha=best_alpha, user_normalize=False, checkpoint_epoch=best_epoch,
                         epochs_run=len(selected_rows), **comparison_bpr_config)
    test_models["validation_selected_alpha"] = best_alpha_model, configuration

main_comparison, test_per_user = evaluate_test_models(
    test_models, train_df, val_df, test_df, num_users, num_items
)
main_comparison.to_csv(RESULTS_DIR / "main_comparison.csv", index=False)
display(main_comparison.drop(columns=["seed", "split", "config"]).round(4))

### 4.2 Experiment B: playtime sensitivity (alpha sweep, validation)

We tried $\alpha\in\{0,0.25,0.5,1,2\}$, with the same split, seed and training settings for each run (up to 100 epochs, patience of five). For each $\alpha$ we kept the checkpoint with the best validation NDCG@10, and if two values of $\alpha$ tied we would pick the smaller one. This choice was made before looking at the test set.

Validation NDCG@10 dropped every time $\alpha$ increased, from 0.2364 at $\alpha=0$ to 0.2329 at $\alpha=0.25$ and down to 0.2023 at $\alpha=2$. So the best choice is $\alpha=0$, which is just standard BPR and is already in the test table.

In [ ]:
alpha_history.to_csv(RESULTS_DIR / "bpr_alpha_sweep.csv", index=False)
best_epochs = alpha_history.groupby("alpha", sort=True)["val_ndcg@10"].idxmax()
alpha_summary = alpha_history.loc[best_epochs, ["alpha", "epoch", "val_ndcg@10"]]
display(alpha_summary.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alpha_summary["alpha"], alpha_summary["val_ndcg@10"], marker="o")
ax.set(xlabel="Playtime weight exponent (alpha)", ylabel="Validation NDCG@10")
ax.set_xticks(alpha_summary["alpha"])
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bpr_alpha_sweep.png", dpi=160)
plt.show()

### 4.3 Experiment C: user activity groups

We sorted users by their total training playtime (ties broken by user ID) and split them into the bottom 25%, the middle 50% and the top 25%. This shows whether playtime weighting helps light or heavy players more.

Standard BPR had the highest test NDCG@10 of the three BPR models in every group. Its lead over log-weighted BPR shrinks as users play more, from 0.0250 in the bottom 25% to 0.0202 in the middle 50% and 0.0159 in the top 25%. So weighting hurts least for heavy players, but it never beats standard BPR. Since heavy players also tend to own more games, the check below tests whether this pattern is really about playtime or about library size.

In [ ]:
def activity_group_metrics(train_df, test_per_user, model_names):
    """Macro-average model metrics within training-playtime groups."""
    hours = train_df.groupby("user_id", sort=True)["playtime_hours"].sum()
    ordered_users = hours.sort_values(kind="stable").index
    quarter = len(ordered_users) // 4
    labels = ["Bottom 25%", "Middle 50%", "Top 25%"]
    sizes = [quarter, len(ordered_users) - 2 * quarter, quarter]
    groups = pd.Series(
        pd.Categorical(np.repeat(labels, sizes), categories=labels, ordered=True),
        index=ordered_users,
        name="activity_group",
    )
    rows = []
    for model_name in model_names:
        users = test_per_user[model_name].join(groups, on="user_id")
        summary = users.groupby("activity_group", observed=True).agg(
            n_users=("user_id", "size"),
            **{metric: (metric, "mean") for metric in ("recall@10", "recall@20", "ndcg@10", "auc")},
        ).reset_index()
        rows.append(summary.assign(model=model_name, seed=SEED, split="test"))
    return pd.concat(rows, ignore_index=True)


activity_results = activity_group_metrics(train_df, test_per_user, bpr_variants)
activity_results.to_csv(RESULTS_DIR / "bpr_user_activity_test.csv", index=False)
display(activity_results.drop(columns=["seed", "split"]).round(4))

**Playtime or library size?** Users who play more also tend to own more games, so the pattern above could come from either one. To separate them, we repeat the comparison with users grouped by how many training games they own. If the gap between weighted and standard BPR changes just as much across library-size groups as across playtime groups, we cannot say the pattern is caused by playtime.

In [ ]:
# Is the pattern about playtime or about library size? Total playtime and library size are
# correlated (Section 2.4), so repeat the grouping by number of training games, using the same
# rule as activity_group_metrics (stable sort with ties by user ID, quarter = n // 4).
def quartile_groups(values, name):
    ordered_users = values.sort_values(kind="stable").index
    quarter = len(ordered_users) // 4
    labels = ["Bottom 25%", "Middle 50%", "Top 25%"]
    sizes = [quarter, len(ordered_users) - 2 * quarter, quarter]
    return pd.Series(pd.Categorical(np.repeat(labels, sizes), categories=labels, ordered=True),
                     index=ordered_users, name=name).sort_index()


by_user = train_df.groupby("user_id", sort=True)
playtime_group = quartile_groups(by_user["playtime_hours"].sum(), "playtime group")
library_group = quartile_groups(by_user["item_id"].size(), "library-size group")

# How often do the two groupings agree?
display(pd.crosstab(playtime_group, library_group))
print(f"Users in the same group under both groupings: {(playtime_group == library_group).mean():.0%}")

# NDCG@10 of each BPR variant under both groupings, and the weighted-minus-standard gap.
ndcg_by_group = {}
for grouping in (playtime_group, library_group):
    for model_name in bpr_variants:
        per_user = test_per_user[model_name].join(grouping, on="user_id")
        ndcg_by_group[(grouping.name, model_name)] = (
            per_user.groupby(grouping.name, observed=True)["ndcg@10"].mean())
ndcg_by_group = pd.DataFrame(ndcg_by_group).T
ndcg_by_group.index.names = ["grouping", "model"]
for weighted in ("log_weighted_bpr", "user_normalized_bpr"):
    for grouping in (playtime_group.name, library_group.name):
        ndcg_by_group.loc[(grouping, f"{weighted} - standard"), :] = (
            ndcg_by_group.loc[(grouping, weighted)] - ndcg_by_group.loc[(grouping, "standard_bpr")])
display(ndcg_by_group.sort_index().round(4))

# Share of held-out test games with zero recorded playtime in each group, pooled over held-out
# games. Section 4.5 shows that playtime weighting mainly loses ground on these games.
never_played = test_df.assign(never_played=test_df["playtime_hours"] == 0)
never_played_share = pd.DataFrame({
    grouping.name: never_played.join(grouping, on="user_id")
                               .groupby(grouping.name, observed=True)["never_played"].mean()
    for grouping in (playtime_group, library_group)})
never_played_share.index.name = "group"
display(never_played_share.round(3))

Only 57% of users land in the same group under both groupings, so the two can be told apart. The gap between log-weighted and standard BPR narrows from the bottom to the top quarter under both, but more by playtime (from −0.025 to −0.016 NDCG@10) than by library size (from −0.022 to −0.019). So the pattern follows playtime more closely than library size.

This fits the finding in Section 4.5 that weighting mainly hurts on held-out games the user never played. Light players have the most never-played held-out games (38% in the bottom playtime quarter vs 34% in the top), so weighting costs them the most. Users with large libraries have the most never-played held-out games of all (40% in the top library-size quarter vs 25% in the bottom), which is why the gap barely shrinks across library-size groups.

### 4.4 Effect of extending the BPR training budget (validation)

With our original limit of 30 epochs, the models were still improving at the end, so we raised the limit to 100 epochs with the same early-stopping rule. The table compares each model's best validation NDCG@10 within the first 30 epochs with its best over the full run. We also re-scored each selected checkpoint with the shared evaluator on all 56,457 validation users to confirm the numbers.

All three models improved. Standard BPR went from 0.2138 (epoch 30) to 0.2364 (epoch 91) and stayed the best of the three. Sections 4.1 and 4.3 use the best checkpoints from these longer runs.

In [ ]:
extended_summaries, extended_histories = [], []
for name, (model, history) in bpr_runs.items():
    alpha, _ = bpr_variants[name]
    best = max(history, key=lambda row: row["val_ndcg@10"])
    previous = max(history[:30], key=lambda row: row["val_ndcg@10"])
    metrics = evaluate(model, val_df, [train_df], num_users, num_items, batch_size=512)
    np.testing.assert_allclose(metrics["ndcg@10"], best["val_ndcg@10"], rtol=0, atol=1e-12)
    run = dict(model=name, split="validation", alpha=alpha, **comparison_bpr_config,
               train_interactions=len(train_df), validation_interactions=len(val_df),
               torch_threads=torch.get_num_threads())
    extended_summaries.append({
        **run,
        **metrics,
        "epoch": best["epoch"],
        "epochs_run": len(history),
        "early_stopped": len(history) < comparison_bpr_config["epochs"],
        "previous_best_epoch": previous["epoch"],
        "previous_ndcg@10": previous["val_ndcg@10"],
        "ndcg@10_gain": metrics["ndcg@10"] - previous["val_ndcg@10"],
    })
    extended_histories.append(pd.DataFrame(history).assign(**run))

extended_results = pd.DataFrame(extended_summaries)
extended_history = pd.concat(extended_histories, ignore_index=True)
extended_results.to_csv(RESULTS_DIR / "bpr_extended_validation.csv", index=False)
extended_history.to_csv(RESULTS_DIR / "bpr_extended_training_history.csv", index=False)

extended_labels = {
    "standard_bpr": "Standard BPR",
    "log_weighted_bpr": "Log-weighted BPR",
    "user_normalized_bpr": "User-normalized BPR",
}
extended_table = extended_results[
    ["model", "previous_ndcg@10", "ndcg@10", "epoch", "epochs_run"]
].copy()
extended_table["model"] = extended_table["model"].map(extended_labels)
display(extended_table.rename(columns={
    "model": "Model",
    "previous_ndcg@10": "30-epoch validation NDCG@10",
    "ndcg@10": "Extended validation NDCG@10",
    "epoch": "Best epoch",
    "epochs_run": "Stopped at",
}).round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
for (name, label), color in zip(extended_labels.items(), ["#22568b", "#af5927", "#3b7d68"]):
    history = extended_history[extended_history["model"] == name]
    best = history.loc[history["val_ndcg@10"].idxmax()]
    ax.plot(history["epoch"], history["val_ndcg@10"], color=color, label=label)
    ax.scatter(best["epoch"], best["val_ndcg@10"], color=color, zorder=3, s=28)
ax.axvline(30, color="#606060", ls="--", lw=1, label="Previous 30-epoch cap")
ax.set(xlabel="Training epoch", ylabel="Validation NDCG@10")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bpr_extended_validation.png", dpi=160)
plt.show()

### 4.5 Played versus never-played held-out games

About a third of the held-out test games (35%) were never played, but they still count as hits if a model recommends them. Playtime weighting gives these games less weight during training, so if that is why weighting hurts, the weighted models should mainly lose ground on never-played games. For each model, we measure the share of held-out games that reach the user's top 20, separately for played and never-played games. This share is pooled over all held-out games rather than averaged per user, so it differs from Recall@20 in Section 4.1. Test playtime is only used to split the results into the two groups, and no model sees it.

In [ ]:
def heldout_hits_by_playtime(model, k=20, batch_size=512):
    """For each held-out test game, whether it appears in the user's top k (same masking and
    tie-breaking as evaluate). Returns a boolean array aligned with test_df rows."""
    masked = interaction_matrix([train_df, val_df], num_users, num_items)
    users = np.unique(test_df["user_id"].to_numpy())
    top_rows, top_cols = [], []
    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        scores = model.score_users(batch)
        if hasattr(scores, "detach"):
            scores = scores.detach().cpu().numpy()
        scores = np.array(scores, dtype=np.float64)
        scores[masked[batch].toarray().astype(bool)] = -np.inf
        top = np.argsort(-scores, axis=1, kind="stable")[:, :k]
        top_rows.append(np.repeat(batch, k))
        top_cols.append(top.ravel())
    in_top = sp.csr_matrix((np.ones(sum(len(r) for r in top_rows), dtype=np.int8),
                            (np.concatenate(top_rows), np.concatenate(top_cols))),
                           shape=(num_users, num_items))
    return np.asarray(in_top[test_df["user_id"].to_numpy(), test_df["item_id"].to_numpy()]).ravel() > 0


# Played vs never-played held-out games. Test playtime is only used to group the results here;
# no model sees it. Hit rate = share of held-out games of that kind that reach the top 20.
played = test_df["playtime_hours"].to_numpy() > 0
print(f"Held-out test games never played: {(~played).mean():.1%} of {len(played):,}")
playtime_split = []
for name, (model, _) in test_models.items():
    hits = heldout_hits_by_playtime(model, k=20)
    playtime_split.append({"model": name,
                           "hit rate@20 (played)": hits[played].mean(),
                           "hit rate@20 (never played)": hits[~played].mean(),
                           "hit rate@20 (all)": hits.mean()})
playtime_split = pd.DataFrame(playtime_split).set_index("model")
playtime_split.to_csv(RESULTS_DIR / "heldout_hits_by_playtime.csv")
display(playtime_split.round(4))

**Playtime weighting does what it is designed to do, but the task rewards something else.** On held-out games the user *played*, both weighted models beat standard BPR (hit rate@20 of 0.297 and 0.293 against 0.281). On held-out games the user *never played*, they fall far behind (0.163 and 0.158 against 0.237). Since 35% of the test games were never played, the loss there is bigger than the gain on played games, so the weighted models lose overall.

In other words, weighting ranks the games a user actually played higher, but ranks games they only own lower. Which is better depends on the goal. For predicting ownership, as we evaluate here, weighting is worse. For recommending games people will actually play, it may be better, and evaluating only on played games would be a natural next experiment. Popularity shows an extreme version of the same pattern, finding played games far more often than never-played ones (0.189 against 0.071), while Jaccard is close to balanced (0.230 against 0.206).

## 5. Related work and limitations

### How this dataset has been used

- **Pathak, Gupta and McAuley (2017)** collected this dataset by crawling members of the Australian "GameAus" Steam community, along with Steam's bundle offers, to study bundle recommendation. Their first stage is item-level BPR with matrix factorization, evaluated by AUC, which is close to our standard BPR. They note that game popularity follows a power law, and that sampling negatives uniformly therefore biases the model toward popular games. To fix this, they sample negatives so that they have the same popularity distribution as the positives. [Generating and Personalizing Bundle Recommendations on Steam](https://cseweb.ucsd.edu/~jmcauley/pdfs/sigir17.pdf)
- **Wan and McAuley (2018)** use a filtered version of the same Steam data to model *monotonic behavior chains*, where a stronger action implies the weaker ones. On Steam, recommending a game implies reviewing it, which implies playing it, which implies buying it. Their model, chainRec, predicts each stage of the chain instead of treating all interactions the same, and is evaluated with AUC and NDCG. They report that about 36% of purchased games were never played. [Item Recommendation on Monotonic Behavior Chains](https://cseweb.ucsd.edu/~jmcauley/pdfs/recsys18b.pdf)
- **Kang and McAuley (2018)** use the larger Version 2 Steam review data for sequential recommendation, predicting a user's next game from the order of their reviews. Our snapshot has no timestamps, so we cannot use this setting. [Self-Attentive Sequential Recommendation](https://arxiv.org/abs/1808.09781)

### How prior work approached the task

- **Rendle et al. (2009)** introduce BPR for ranking from implicit feedback. For each user, the model is trained to score an observed item higher than a sampled, unobserved item. We apply this to Steam libraries using dot-product embeddings and one sampled game per owned game each epoch. [BPR: Bayesian Personalized Ranking from Implicit Feedback](https://arxiv.org/abs/1205.2618)
- **Hu, Koren and Volinsky (2008)** use the amount of feedback (here, TV watching time) to set confidence weights in a squared-error model, and also describe a logarithmic version. We use log-scaled training playtime to weight the BPR loss for each owned game, and also test keeping each user's total weight fixed while changing the relative weights of their games. [Collaborative Filtering for Implicit Feedback Datasets](https://yifanhu.net/PUB/cf.pdf)
- **Lerche and Jannach (2014)** extend BPR to graded implicit feedback, so that stronger signals are ranked above weaker ones instead of being treated as equal positives. On two shopping datasets their time-based variant improves on BPR, and a rating-based variant does on MovieLens, but their count-based variant does worse than BPR on one of the shopping datasets. Our weighting is a related but simpler idea, so we do not claim it as a new method. [Using Graded Implicit Feedback for Bayesian Personalized Ranking](https://web-ainf.aau.at/pub/jannach/files/Conference_RecSys2014a.pdf)

### How our results compare

- **BPR versus the baselines.** Standard BPR was the best model on every test metric (NDCG@10 0.279, AUC 0.977), ahead of item-based Jaccard (0.270, 0.961) and popularity (0.158, 0.950). Its lead over Jaccard is small, though, and Jaccard needs no training.
- **AUC rewards popularity.** Popularity alone reaches an AUC of 0.950, because most games a user does not own are obscure and easy to rank low. Wan and McAuley see the same on Steam, where popularity reaches an AUC of 0.955 against 0.963 for BPR when predicting which games users recommend. This is why we use NDCG@10 as our main metric, since it separates the models much more clearly (0.158 to 0.279) than AUC does (0.950 to 0.977).
- **Playtime weighting did not help overall.** Hu et al. and Lerche and Jannach report gains from weighting by feedback strength, but every weighted variant we tried scored below standard BPR. Validation NDCG@10 also fell steadily as the weight exponent grew (0.236 at $\alpha=0$ to 0.202 at $\alpha=2$), and longer training did not change this (Section 4.4).
  - *The main reason is what we predict.* We predict ownership, and about 35% of owned games were never played, close to the 36% Wan and McAuley report. Section 4.5 shows that the weighted models rank held-out *played* games better than standard BPR, but held-out *never-played* games much worse. Hu et al. only count shows watched for at least half their length, and Lerche and Jannach only count purchases (or above-average ratings on MovieLens), so both evaluate on the stronger signal they weight by. We evaluate on ownership, the weakest signal, which playtime weighting deliberately gives less weight to.
  - *Playtime also reflects the type of game.* Multiplayer and open-ended games collect far more hours than short single-player games a user may like just as much, so the weights partly reflect genre rather than how much a user likes a game. We have not tested this.
  - *A better use of playtime.* Following Wan and McAuley, playing could be treated as a separate, stronger behavior to predict, instead of being used to reweight the ownership objective.

### Limitations

- We used a single seed, so differences of about 0.01 NDCG@10 between models may be within seed-to-seed variation.
- All BPR variants share one training configuration. This keeps the comparison controlled, but the weighted variants might do better with a different learning rate or regularization.
- The split is random rather than by time, and held-out ownership is only a proxy for preference, since a game a user does not own is not necessarily one they would dislike.
- The users come from a single Australian community with public profiles, and the non-game filter relies on game names.